# 05 · Statistical Testing
Hypothesis testing, ANOVA, correlation analysis, and confidence intervals to validate observed differences in channel/campaign performance are statistically significant (not noise).

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

campaigns = pd.read_csv('../data/processed/campaigns_clean.csv', parse_dates=['start_date','end_date'])

## A/B-style Test: Google Ads vs Facebook Ads ROI
Independent two-sample t-test comparing mean ROI between the two largest paid channels.

In [2]:
g = campaigns.loc[campaigns['channel']=='Google Ads', 'roi']
f = campaigns.loc[campaigns['channel']=='Facebook Ads', 'roi']
t_stat, p_val = stats.ttest_ind(g, f, equal_var=False)
print(f'Google mean ROI={g.mean():.3f}, Facebook mean ROI={f.mean():.3f}')
print(f't-statistic={t_stat:.3f}, p-value={p_val:.5f}')

Google mean ROI=19.936, Facebook mean ROI=12.559
t-statistic=5.734, p-value=0.00000


## ANOVA: ROI Across All Channels
One-way ANOVA to test whether mean ROI differs significantly across all 7 channels.

In [3]:
groups = [grp['roi'].values for _, grp in campaigns.groupby('channel')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'F-statistic={f_stat:.2f}, p-value={p_val:.6f}')
model = ols('roi ~ C(channel)', data=campaigns).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

F-statistic=316.66, p-value=0.000000


,sum_sq,df,F,PR(>F)
C(channel),1.455763e+09,6.0,316.658725,0.0
Residual,3.978929e+09,5193.0,NaN,NaN


## Correlation Analysis

In [4]:
num_cols = ['budget','impressions','clicks','ctr','cpc','conversions',
            'conversion_rate','revenue','cost','roi','roas']
corr = campaigns[num_cols].corr()
corr['roi'].sort_values(ascending=False)

roi                1.000000
roas               1.000000
revenue            0.551576
conversions        0.467430
ctr                0.407346
clicks             0.403787
conversion_rate    0.285616
impressions        0.114130
cpc               -0.253375
cost              -0.278450
budget            -0.297220
Name: roi, dtype: float64

## Linear Regression: Predictors of ROI

In [5]:
X = campaigns[['ctr','conversion_rate','cpc']]
X = sm.add_constant(X)
y = campaigns['roi']
lin_model = sm.OLS(y, X).fit()
print(lin_model.summary())

                            OLS Regression Results                            
Dep. Variable:                    roi   R-squared:                       0.187
Model:                            OLS   Adj. R-squared:                  0.187
Method:                 Least Squares   F-statistic:                     398.4
Date:                Sun, 12 Jul 2026   Prob (F-statistic):          6.20e-233
Time:                        19:50:06   Log-Likelihood:                -42875.
No. Observations:                5200   AIC:                         8.576e+04
Df Residuals:                    5196   BIC:                         8.578e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            -538.2619     42.246    -

## Logistic Regression: Probability Campaign is 'Successful' (ROI > median)

In [6]:
campaigns['success'] = (campaigns['roi'] > campaigns['roi'].median()).astype(int)
X = campaigns[['ctr','conversion_rate','cpc']]
X = sm.add_constant(X)
y = campaigns['success']
logit_model = sm.Logit(y, X).fit(disp=0)
print(logit_model.summary())

                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 5200
Model:                          Logit   Df Residuals:                     5196
Method:                           MLE   Df Model:                            3
Date:                Sun, 12 Jul 2026   Pseudo R-squ.:                  0.4608
Time:                        19:50:06   Log-Likelihood:                -1943.5
converged:                       True   LL-Null:                       -3604.4
Covariance Type:            nonrobust   LLR p-value:                     0.000
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -3.3309      0.176    -18.922      0.000      -3.676      -2.986
ctr                55.5936      2.713     20.492      0.000      50.276      60.911
conversion_rate    55.7654      

## 95% Confidence Interval for Mean ROI by Channel

In [7]:
def ci_95(series):
    n = len(series); mean = series.mean(); sem = stats.sem(series)
    h = sem * stats.t.ppf(0.975, n-1)
    return pd.Series({'mean_roi': mean, 'ci_lower': mean-h, 'ci_upper': mean+h})

campaigns.groupby('channel')['roi'].apply(ci_95).unstack().round(3).sort_values('mean_roi', ascending=False)

,mean_roi,ci_lower,ci_upper
channel,,,
Referral,1767.536,1554.554,1980.517
Organic Search,938.368,823.143,1053.592
Email Marketing,301.176,260.955,341.397
Affiliate,40.533,30.661,50.404
Google Ads,19.936,17.775,22.096
Facebook Ads,12.559,11.254,13.863
LinkedIn,5.215,4.334,6.096


### Statistical Conclusions
- Channel is a statistically significant driver of ROI (ANOVA p < 0.05); differences observed in EDA are not attributable to random noise.
- Conversion rate and CTR are positively associated with ROI; CPC is negatively associated, consistent with marketing economics intuition.
- Confidence intervals confirm Email Marketing, Organic Search, and Referral channels have significantly higher mean ROI than paid social/search channels.